# 01 — Scrape Guardian News API
Fetches articles tagged with AI, geopolitics, energy, climate, cyber, health, and macro-risk keywords from The Guardian API.
Output: `data/raw_articles.csv`

In [ ]:
import requests
import pandas as pd
import time
import os

os.makedirs('data', exist_ok=True)

API_KEY = '###'
BASE_URL = 'https://content.guardianapis.com/search'

# ── Query ────────────────────────────────────────────────────
# Broad query: AI + geopolitics + energy + macro + climate + cyber
# + public health + supply chain + governance + commodities.
QUERY = (
    '"artificial intelligence" OR "machine learning" OR "generative AI" OR "ChatGPT" OR "OpenAI" OR '
    '"Israel" OR "Iran" OR "Middle East" OR "war" OR "sanctions" OR '
    '"oil price" OR "OPEC" OR "natural gas" OR "LNG" OR "Strait of Hormuz" OR '
    '"inflation" OR "interest rates" OR "recession" OR "central bank" OR '
    '"climate change" OR "global warming" OR "extreme weather" OR "flood" OR "heatwave" OR "drought" OR '
    '"cyberattack" OR "ransomware" OR "data breach" OR "critical infrastructure" OR '
    '"WHO" OR "outbreak" OR "pandemic" OR "vaccine" OR '
    '"supply chain" OR "shipping disruption" OR "port congestion" OR "semiconductor" OR '
    '"election" OR "protest" OR "governance" OR "tariff" OR "export controls" OR '
    '"rare earth" OR "lithium" OR "copper" OR "uranium" OR "food security" OR "water scarcity"'
)

FROM_DATE = '2021-01-01'
TO_DATE   = '2026-04-18'
PAGE_SIZE = 200   # API maximum

# ── Section allow-list ───────────────────────────────────────
# Only fetch from sections where AI coverage is substantive.
# The Guardian API supports `section=` as a comma-separated filter.
# Avoids sports, entertainment, lifestyle pulling in stray matches.
ALLOWED_SECTIONS = [
    'technology', 'business', 'science', 'politics',
    'world', 'us-news', 'uk-news',
    'media', 'environment', 'society', 'global-development',
    'education', 'law', 'money', 'news', 'opinion',
    'the-filter',
]

# ── Topic taxonomy + relevance filter (post-fetch) ─────────
# Multi-label taxonomy for Phase 1 expansion.
TOPIC_KEYWORDS = {
    'ai': [
        'artificial intelligence', ' ai ', 'ai,', 'machine learning',
        'generative ai', 'large language model', 'llm', 'chatgpt',
        'gpt-4', 'gpt-3', 'openai', 'deep learning', 'neural network',
        'ai regulation', 'ai safety', 'ai act', 'deepmind', 'ai bias',
        'ai ethics', 'algorithm', 'robot', 'automation', 'anthropic',
        'gemini', 'copilot', 'claude'
    ],
    'geopolitics_conflict': [
        'israel', 'iran', 'middle east', 'gulf', 'hormuz',
        'war', 'ceasefire', 'missile', 'military', 'sanctions',
        'navy', 'airstrike', 'drone strike', 'troops', 'border clash'
    ],
    'energy_markets': [
        'oil', 'crude', 'opec', 'energy crisis', 'shipping disruption',
        'strait of hormuz', 'brent', 'natural gas', 'lng',
        'refinery', 'diesel', 'electricity prices'
    ],
    'macroeconomy': [
        'inflation', 'interest rate', 'interest rates', 'recession',
        'central bank', 'federal reserve', 'ecb', 'unemployment', 'gdp',
        'bond yields', 'debt crisis', 'currency volatility'
    ],
    'climate_weather': [
        'weather', 'extreme weather', 'flood', 'heatwave', 'drought',
        'climate change', 'global warming', 'carbon emissions',
        'renewable energy', 'wildfire', 'storm', 'hurricane'
    ],
    'public_health': [
        'outbreak', 'pandemic', 'epidemic', 'vaccine', 'who',
        'hospital strain', 'infectious disease', 'public health emergency'
    ],
    'cybersecurity': [
        'cyberattack', 'cyber attack', 'ransomware', 'data breach',
        'critical infrastructure hack', 'malware', 'ddos', 'zero-day'
    ],
    'supply_chain': [
        'supply chain', 'port congestion', 'shipping delays',
        'container rates', 'freight costs', 'semiconductor shortage',
        'factory shutdown'
    ],
    'food_water_security': [
        'food security', 'food prices', 'crop failure', 'grain exports',
        'water scarcity', 'reservoir levels', 'famine'
    ],
    'finance_banking': [
        'bank stress', 'bank run', 'liquidity crisis', 'credit crunch',
        'sovereign debt', 'default risk'
    ],
    'elections_governance': [
        'election', 'vote', 'parliament', 'governance', 'protest',
        'constitutional crisis', 'policy shift'
    ],
    'defense_security': [
        'defense', 'arms transfer', 'naval deployment', 'deterrence',
        'security pact', 'military exercise'
    ],
    'trade_industry': [
        'tariff', 'export controls', 'trade restrictions',
        'industrial policy', 'reshoring', 'nearshoring'
    ],
    'labor_social': [
        'strike', 'labor dispute', 'layoffs', 'migration pressure',
        'social unrest', 'cost of living crisis'
    ],
    'natural_disasters': [
        'earthquake', 'tsunami', 'landslide', 'cyclone', 'wildfire',
        'humanitarian emergency', 'disaster relief'
    ],
    'technology_policy': [
        'data protection', 'privacy law', 'antitrust',
        'platform regulation', 'digital services act', 'online safety law'
    ],
    'commodities_metals': [
        'rare earth', 'lithium', 'copper', 'nickel', 'uranium',
        'commodity shock', 'mining disruption'
    ],
}

RELEVANCE_TERMS = sorted({kw for kws in TOPIC_KEYWORDS.values() for kw in kws})


def detect_topics(headline: str, body: str) -> list:
    """Return all topic labels matched in headline or lead paragraph."""
    text = (str(headline) + ' ' + str(body)[:3000]).lower()
    return [
        topic
        for topic, keywords in TOPIC_KEYWORDS.items()
        if any(term in text for term in keywords)
    ]


def is_topic_relevant(headline: str, body: str) -> bool:
    """Return True if at least one target topic term is found."""
    text = (str(headline) + ' ' + str(body)[:3000]).lower()
    return any(term in text for term in RELEVANCE_TERMS)


def primary_topic(matched_topics: list) -> str:
    """Pick a stable primary topic based on taxonomy order."""
    if not matched_topics:
        return 'other'
    for topic in TOPIC_KEYWORDS.keys():
        if topic in matched_topics:
            return topic
    return matched_topics[0]

In [7]:
def fetch_page(page: int) -> dict:
    params = {
        'q':          QUERY,
        'from-date':  FROM_DATE,
        'to-date':    TO_DATE,
        'page-size':  PAGE_SIZE,
        'page':       page,
        'show-fields': 'bodyText,headline,wordcount',
        'section':    '|'.join(ALLOWED_SECTIONS),  # server-side section filter
        'api-key':    API_KEY,
    }
    r = requests.get(BASE_URL, params=params)
    r.raise_for_status()
    return r.json()['response']


def parse_results(results: list) -> list:
    rows = []
    for item in results:
        fields = item.get('fields', {})
        headline = fields.get('headline', '')
        body     = fields.get('bodyText', '')

        # Skip articles where target-topic terms don't appear in headline or lead
        if not is_topic_relevant(headline, body):
            continue

        matched_topics = detect_topics(headline, body)

        rows.append({
            'id':            item.get('id'),
            'date':          item.get('webPublicationDate', '')[:10],
            'section':       item.get('sectionName'),
            'headline':      headline,
            'body':          body,
            'wordcount':     fields.get('wordcount', 0),
            'url':           item.get('webUrl'),
            'topics':        '|'.join(matched_topics),
            'topic_count':   len(matched_topics),
            'primary_topic': primary_topic(matched_topics),
        })
    return rows

In [8]:
import sys

def fetch_page_with_retry(page: int, retries: int = 3, backoff: float = 5.0) -> dict:
    """Fetch a page with exponential backoff on 5xx errors."""
    for attempt in range(retries):
        try:
            return fetch_page(page)
        except requests.HTTPError as e:
            if e.response.status_code >= 500 and attempt < retries - 1:
                wait = backoff * (2 ** attempt)
                print(f'  [503] Page {page} failed, retrying in {wait:.0f}s...', flush=True)
                time.sleep(wait)
            else:
                raise

# ── Paginate ─────────────────────────────────────────────────
all_rows = []
first = fetch_page_with_retry(1)
total_pages = first['pages']
total_articles = first['total']
print(f'API reports {total_articles:,} articles across {total_pages} pages')

all_rows.extend(parse_results(first['results']))

for page in range(2, total_pages + 1):
    try:
        resp = fetch_page_with_retry(page)
        all_rows.extend(parse_results(resp['results']))
    except Exception as e:
        print(f'  Page {page} failed permanently: {e}', file=sys.stderr)

    if page % 10 == 0 or page == total_pages:
        print(f'  Page {page}/{total_pages} — kept so far: {len(all_rows):,}', flush=True)

    time.sleep(5)

print(f'\nDone. Raw rows after relevance filter: {len(all_rows):,}')

API reports 198,191 articles across 991 pages
  Page 10/991 — kept so far: 1,998
  Page 20/991 — kept so far: 3,997
  Page 30/991 — kept so far: 5,994
  Page 40/991 — kept so far: 7,989
  Page 50/991 — kept so far: 9,983
  Page 60/991 — kept so far: 11,973
  Page 70/991 — kept so far: 13,966
  Page 80/991 — kept so far: 15,956
  Page 90/991 — kept so far: 17,946
  Page 100/991 — kept so far: 19,931
  Page 110/991 — kept so far: 21,917
  Page 120/991 — kept so far: 23,904
  Page 130/991 — kept so far: 25,900
  Page 140/991 — kept so far: 27,886
  Page 150/991 — kept so far: 29,868
  Page 160/991 — kept so far: 31,849
  Page 170/991 — kept so far: 33,832
  Page 180/991 — kept so far: 35,814
  Page 190/991 — kept so far: 37,802


  Page 191 failed permanently: 400 Client Error: Bad Request for url: https://content.guardianapis.com/search?q=%22artificial+intelligence%22+OR+%22machine+learning%22+OR+%22generative+AI%22+OR+%22ChatGPT%22+OR+%22OpenAI%22+OR+%22Israel%22+OR+%22Iran%22+OR+%22Middle+East%22+OR+%22war%22+OR+%22sanctions%22+OR+%22oil+price%22+OR+%22OPEC%22+OR+%22natural+gas%22+OR+%22LNG%22+OR+%22Strait+of+Hormuz%22+OR+%22inflation%22+OR+%22interest+rates%22+OR+%22recession%22+OR+%22central+bank%22+OR+%22climate+change%22+OR+%22global+warming%22+OR+%22extreme+weather%22+OR+%22flood%22+OR+%22heatwave%22+OR+%22drought%22+OR+%22cyberattack%22+OR+%22ransomware%22+OR+%22data+breach%22+OR+%22critical+infrastructure%22+OR+%22WHO%22+OR+%22outbreak%22+OR+%22pandemic%22+OR+%22vaccine%22+OR+%22supply+chain%22+OR+%22shipping+disruption%22+OR+%22port+congestion%22+OR+%22semiconductor%22+OR+%22election%22+OR+%22protest%22+OR+%22governance%22+OR+%22tariff%22+OR+%22export+controls%22+OR+%22rare+earth%22+OR+%22lithium%22+

  Page 200/991 — kept so far: 37,802


  Page 200 failed permanently: 400 Client Error: Bad Request for url: https://content.guardianapis.com/search?q=%22artificial+intelligence%22+OR+%22machine+learning%22+OR+%22generative+AI%22+OR+%22ChatGPT%22+OR+%22OpenAI%22+OR+%22Israel%22+OR+%22Iran%22+OR+%22Middle+East%22+OR+%22war%22+OR+%22sanctions%22+OR+%22oil+price%22+OR+%22OPEC%22+OR+%22natural+gas%22+OR+%22LNG%22+OR+%22Strait+of+Hormuz%22+OR+%22inflation%22+OR+%22interest+rates%22+OR+%22recession%22+OR+%22central+bank%22+OR+%22climate+change%22+OR+%22global+warming%22+OR+%22extreme+weather%22+OR+%22flood%22+OR+%22heatwave%22+OR+%22drought%22+OR+%22cyberattack%22+OR+%22ransomware%22+OR+%22data+breach%22+OR+%22critical+infrastructure%22+OR+%22WHO%22+OR+%22outbreak%22+OR+%22pandemic%22+OR+%22vaccine%22+OR+%22supply+chain%22+OR+%22shipping+disruption%22+OR+%22port+congestion%22+OR+%22semiconductor%22+OR+%22election%22+OR+%22protest%22+OR+%22governance%22+OR+%22tariff%22+OR+%22export+controls%22+OR+%22rare+earth%22+OR+%22lithium%22+

  Page 210/991 — kept so far: 37,802


...

  Page 260/991 — kept so far: 37,802


  Page 260 failed permanently: 400 Client Error: Bad Request for url: https://content.guardianapis.com/search?q=%22artificial+intelligence%22+OR+%22machine+learning%22+OR+%22generative+AI%22+OR+%22ChatGPT%22+OR+%22OpenAI%22+OR+%22Israel%22+OR+%22Iran%22+OR+%22Middle+East%22+OR+%22war%22+OR+%22sanctions%22+OR+%22oil+price%22+OR+%22OPEC%22+OR+%22natural+gas%22+OR+%22LNG%22+OR+%22Strait+of+Hormuz%22+OR+%22inflation%22+OR+%22interest+rates%22+OR+%22recession%22+OR+%22central+bank%22+OR+%22climate+change%22+OR+%22global+warming%22+OR+%22extreme+weather%22+OR+%22flood%22+OR+%22heatwave%22+OR+%22drought%22+OR+%22cyberattack%22+OR+%22ransomware%22+OR+%22data+breach%22+OR+%22critical+infrastructure%22+OR+%22WHO%22+OR+%22outbreak%22+OR+%22pandemic%22+OR+%22vaccine%22+OR+%22supply+chain%22+OR+%22shipping+disruption%22+OR+%22port+congestion%22+OR+%22semiconductor%22+OR+%22election%22+OR+%22protest%22+OR+%22governance%22+OR+%22tariff%22+OR+%22export+controls%22+OR+%22rare+earth%22+OR+%22lithium%22+

  Page 270/991 — kept so far: 37,802


  Page 270 failed permanently: 400 Client Error: Bad Request for url: https://content.guardianapis.com/search?q=%22artificial+intelligence%22+OR+%22machine+learning%22+OR+%22generative+AI%22+OR+%22ChatGPT%22+OR+%22OpenAI%22+OR+%22Israel%22+OR+%22Iran%22+OR+%22Middle+East%22+OR+%22war%22+OR+%22sanctions%22+OR+%22oil+price%22+OR+%22OPEC%22+OR+%22natural+gas%22+OR+%22LNG%22+OR+%22Strait+of+Hormuz%22+OR+%22inflation%22+OR+%22interest+rates%22+OR+%22recession%22+OR+%22central+bank%22+OR+%22climate+change%22+OR+%22global+warming%22+OR+%22extreme+weather%22+OR+%22flood%22+OR+%22heatwave%22+OR+%22drought%22+OR+%22cyberattack%22+OR+%22ransomware%22+OR+%22data+breach%22+OR+%22critical+infrastructure%22+OR+%22WHO%22+OR+%22outbreak%22+OR+%22pandemic%22+OR+%22vaccine%22+OR+%22supply+chain%22+OR+%22shipping+disruption%22+OR+%22port+congestion%22+OR+%22semiconductor%22+OR+%22election%22+OR+%22protest%22+OR+%22governance%22+OR+%22tariff%22+OR+%22export+controls%22+OR+%22rare+earth%22+OR+%22lithium%22+

KeyboardInterrupt: 

In [9]:
df = pd.DataFrame(all_rows)
df['date'] = pd.to_datetime(df['date'])
df.drop_duplicates(subset='id', inplace=True)

print(f'\nFinal dataset: {len(df)} articles')
df.head()


Final dataset: 37299 articles


,id,date,section,headline,body,wordcount,url,topics,topic_count,primary_topic
0,business/2026/mar/02/middle-east-crisis-oil-pr...,2026-03-02,Business,Middle East crisis pushes up oil prices – and ...,The impact of the deadly and unpredictable con...,684,https://www.theguardian.com/business/2026/mar/...,ai|geopolitics_conflict|energy_markets|macroec...,6,ai
1,world/2026/apr/08/will-shipping-in-the-strait-...,2026-04-08,World news,Will shipping in the strait of Hormuz – and oi...,"If the US-Israeli ceasefire with Iran holds, i...",1361,https://www.theguardian.com/world/2026/apr/08/...,geopolitics_conflict|energy_markets|public_health,3,geopolitics_conflict
2,business/2026/mar/01/oil-price-surge-iran-us-i...,2026-03-01,Business,Oil price expected to surge after Iran strikes...,The price of oil is expected to soar on Monday...,870,https://www.theguardian.com/business/2026/mar/...,geopolitics_conflict|energy_markets|public_hea...,5,geopolitics_conflict
3,business/2026/mar/06/iran-war-pushes-oil-price...,2026-03-06,Business,"Iran war pushes oil price above $90, threateni...",The Iran conflict has driven the oil price pas...,720,https://www.theguardian.com/business/2026/mar/...,geopolitics_conflict|energy_markets|macroecono...,5,geopolitics_conflict
4,business/live/2026/mar/02/oil-price-us-israel-...,2026-03-02,Business,FTSE 100 share index records biggest fall sinc...,"Time for a recap: Global energy, bond and equi...",7440,https://www.theguardian.com/business/live/2026...,geopolitics_conflict|energy_markets|macroecono...,6,geopolitics_conflict


In [10]:
df = pd.DataFrame(all_rows)
df['date'] = pd.to_datetime(df['date'])
df.drop_duplicates(subset='id', inplace=True)

# Quality filter: drop very short or empty articles
df['wordcount'] = pd.to_numeric(df['wordcount'], errors='coerce').fillna(0).astype(int)
df = df[(df['wordcount'] >= 100) & (df['body'].str.strip() != '')]
df = df.reset_index(drop=True)

print(f'Final dataset: {len(df):,} articles')
print(f'\nSection distribution:')
print(df['section'].value_counts().head(20).to_string())
print(f'\nPrimary topic distribution:')
print(df['primary_topic'].value_counts().to_string())
print(f'\nDate range: {df["date"].min().date()} → {df["date"].max().date()}')

df.to_csv('data/raw_articles.csv', index=False)
print('\nSaved: data/raw_articles.csv')

# Delete stale embedding cache — corpus has changed, must re-embed in NB 02
import os
if os.path.exists('data/embeddings.npy'):
    os.remove('data/embeddings.npy')
    print('Deleted stale data/embeddings.npy — will be regenerated in NB 02')

Final dataset: 37,259 articles

Section distribution:
section
World news            10202
US news                6867
Australia news         4777
Business               4491
Politics               2384
Environment            2160
Technology             1670
UK news                1450
Society                 762
Global development      609
News                    462
Media                   448
Money                   369
Education               237
Science                 226
Law                     145

Primary topic distribution:
primary_topic
geopolitics_conflict    26024
ai                       3772
public_health            3462
macroeconomy             1371
climate_weather          1114
energy_markets            977
elections_governance      393
trade_industry             67
supply_chain               40
commodities_metals         12
cybersecurity               7
defense_security            5
labor_social                5
food_water_security         4
natural_disasters          